In [1]:
from ucimlrepo import fetch_ucirepo
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import plotly.express as px


In [2]:
wine_quality = fetch_ucirepo(name='Wine Quality')
wine_quality_data = wine_quality['data']['original']

In [3]:
seed = 42 #if it must be 42 :P

wine_X = wine_quality_data.drop(columns = ['quality']) # drop quality
wine_X['color'] = (wine_X['color'] == "red").astype(float) # make color a number instead of a string
scaler = StandardScaler()
wine_X = scaler.fit_transform(wine_X) # scaling values
num_features = wine_X.shape[1]

wine_y = wine_quality_data['quality']
min_quality = wine_y.min()
num_classes = wine_y.max() - min_quality + 1
wine_y -= min_quality

wine_X_train_val, wine_X_test, wine_y_train_val, wine_y_test = train_test_split(
    wine_X, wine_y, test_size=0.2, random_state=42
)

wine_X_tr, wine_X_val, wine_y_tr, wine_y_val = train_test_split(
    wine_X_train_val, wine_y_train_val, test_size=0.25, random_state=42
)
print(wine_X_tr.shape, wine_X_val.shape, wine_y_tr.shape, wine_y_val.shape)


(3897, 12) (1300, 12) (3897,) (1300,)


In [4]:
wine_X_tr = torch.tensor(wine_X_tr, dtype=torch.float32)
wine_X_val = torch.tensor(wine_X_val, dtype=torch.float32)
wine_X_test = torch.tensor(wine_X_test, dtype=torch.float32)

wine_y_tr = torch.tensor(wine_y_tr, dtype=torch.long)
wine_y_val = torch.tensor(wine_y_val.values, dtype=torch.long)
wine_y_test = torch.tensor(wine_y_test.values, dtype=torch.long)

In [5]:
dataset = torch.utils.data.TensorDataset(wine_X_tr, wine_y_tr)
trainloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)


In [6]:
def get_model(hidden_dims):
    layer_dims = [num_features, *hidden_dims, num_classes]
    layers = []
    for i in range(len(layer_dims)-2):
        layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))
    return nn.Sequential(*layers)

def accuracy(model, features, labels):
    with torch.no_grad():
        return (torch.argmax(model(features), dim=1) == labels).sum().item() / len(labels)

def train(hidden_dims, epochs, **kwargs):
    print(f"training with hidden_dims: {hidden_dims}, epochs: {epochs}, and kwargs: {kwargs}")

    model = get_model(hidden_dims)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), **kwargs)

    train_accuracy = []
    val_accuracy = []

    log_interval = 100
    for epoch in range(epochs):
        model.train()
        for _, data in enumerate(trainloader, 0):
            inputs, batch_labels = data
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

        if epoch % log_interval == 0:
            model.eval()
            train_accuracy.append((epoch, accuracy(model, wine_X_tr, wine_y_tr)))
            val_accuracy.append((epoch, accuracy(model, wine_X_val, wine_y_val)))

    return train_accuracy, val_accuracy


In [14]:
epochs = 1500
hidden_dims_array = [[16], [32], [64], [16, 32, 16], [16, 64, 16], [32, 64, 32]]

hidden_dims_train_accuracies = []
hidden_dims_val_accuracies = []
for hidden_dims in hidden_dims_array:
    train_accuracy, val_accuracy = train(hidden_dims, epochs)
    hidden_dims_train_accuracies.append(train_accuracy)
    hidden_dims_val_accuracies.append(val_accuracy)

training with hidden_dims: [16], epochs: 1500, and kwargs: {}
training with hidden_dims: [32], epochs: 1500, and kwargs: {}
training with hidden_dims: [64], epochs: 1500, and kwargs: {}
training with hidden_dims: [16, 32, 16], epochs: 1500, and kwargs: {}
training with hidden_dims: [16, 64, 16], epochs: 1500, and kwargs: {}
training with hidden_dims: [32, 64, 32], epochs: 1500, and kwargs: {}


In [19]:
rows = []
for i, hidden_dims in enumerate(hidden_dims_array):
    for epoch, acc in hidden_dims_val_accuracies[i]:
        rows.append({'epoch': epoch, 'validation accuracy': acc, 'hidden_dims': str(hidden_dims)})
fig = px.line(pd.DataFrame(rows), x='epoch', y='validation accuracy', color='hidden_dims')
fig.show()

rows = []
for i, hidden_dims in enumerate(hidden_dims_array):
    for epoch, acc in hidden_dims_train_accuracies[i]:
        rows.append({'epoch': epoch, 'train accuracy': acc, 'hidden_dims': str(hidden_dims)})
fig = px.line(pd.DataFrame(rows), x='epoch', y='train accuracy', color='hidden_dims')
fig.show()

In [20]:
epochs = 1500
learning_rates = [0.0001, 0.001, 0.01, 0.1, 1]

learning_rates_train_accuracies = []
learning_rates_val_accuracies = []
for learning_rate in learning_rates:
    train_accuracy, val_accuracy = train([32,64,32], epochs, lr=learning_rate)
    learning_rates_train_accuracies.append(train_accuracy)
    learning_rates_val_accuracies.append(val_accuracy)

training with hidden_dims: [32, 64, 32], epochs: 1500, and kwargs: {'lr': 0.0001}
training with hidden_dims: [32, 64, 32], epochs: 1500, and kwargs: {'lr': 0.001}
training with hidden_dims: [32, 64, 32], epochs: 1500, and kwargs: {'lr': 0.01}
training with hidden_dims: [32, 64, 32], epochs: 1500, and kwargs: {'lr': 0.1}
training with hidden_dims: [32, 64, 32], epochs: 1500, and kwargs: {'lr': 1}


In [21]:
rows = []
for i, learning_rate in enumerate(learning_rates):
    for epoch, acc in learning_rates_val_accuracies[i]:
        rows.append({'epoch': epoch, 'validation accuracy': acc, 'learning_rate': learning_rate})
fig = px.line(pd.DataFrame(rows), x='epoch', y='validation accuracy', color='learning_rate')
fig.show()

rows = []
for i, learning_rate in enumerate(learning_rates):
    for epoch, acc in learning_rates_train_accuracies[i]:
        rows.append({'epoch': epoch, 'validation accuracy': acc, 'learning_rate': learning_rate})
fig = px.line(pd.DataFrame(rows), x='epoch', y='validation accuracy', color='learning_rate')
fig.show()